In [40]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F


from transformers import AutoImageProcessor, AutoModel

from PIL import Image

In [3]:
# the goal is build a model consisting of an encoder (a visual encoder), a
# nd an RNN whose job is to predict the next state. I want to use lstm 
# for the vision encoder, I want to start with DinoV2 
# 

- Use an off-the-shelf encoder like Dino to get the representationss corresponding to reference image
- the encoder is supposed to give us the embeddings at rotation angle $\phi$
- the embedding sequence is then fed into the LSTM
- the vision encoder is frozen
- only the lstm is trained


I will be building this model step by step.

In [4]:
# Define the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
# get the encoder to work with 
model_variant = "facebook/dinov2-small"

# load the pretrained image processor and model
processor = AutoImageProcessor.from_pretrained(model_variant)
encoder = AutoModel.from_pretrained(model_variant)


The image processor of type `BitImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
Loading weights: 100%|██████████| 223/223 [00:00<00:00, 1618.60it/s, Materializing param=layernorm.weight]                                 


In [6]:
# the goal is to pass a bunch of images to this encoder and get their embeddings
# get a list of images 
# this is the simplest case here, we 
df_path = "/Users/lshahsha/Desktop/SM_new/png/x/"
metadata_all = pd.read_csv(f"{df_path}/polycubes_object_rotation_metadata.csv")
metadata = metadata_all.loc[(metadata_all["is_mirror"] == 0) & (metadata_all["stim_id"] == 1)]
png_paths = [f"{df_path}/{row['filename']}" for i, row in metadata.iterrows()]
images = [Image.open(p).convert("RGB") for p in png_paths]

# now we preprocess these images before passing them to the encoder
# processor handles: resize (patch-aware), normalize, tensor conversion
inputs = processor(images=images, return_tensors="pt").to(device)

# make sure the model is frozen
encoder.eval()
for p in encoder.parameters():
    p.requires_grad = False

# now we pass the inputs to the encoder
with torch.no_grad():
    # pass the images to the encoder and get rid of cls tokens for now
    outputs = encoder(**inputs)
    embeddings = outputs.last_hidden_state[:, 1:, :]
    cls_embeddings = outputs.last_hidden_state[:, 0, :]



In [31]:
embeddings.shape
cls_embeddings.shape

torch.Size([19, 384])

This just takes in all the views of one single object. I need to consider a case where we have multiple objects andd then sort it out accordingly so that the outout is ready to be fed into the lstm

In [29]:
# now let's pass these embeddings to an lstm
# before that, NOTE that the batch here should not be 19, 19 are the time points (rotation angles)
# so this is not batch
# this needs to be the second dimension (equivalent to T)

df_path = "/Users/lshahsha/Desktop/SM_new/png/x"
metadata_all = pd.read_csv(f"{df_path}/polycubes_object_rotation_metadata.csv")
metadata = metadata_all.loc[(metadata_all["is_mirror"] == 0) & (metadata_all["stim_id"] <= 5)]

# here we need to make sure that the dataframe is sorted in a specific way
# it is sorted, but can't afford to be wrong here 
# so I'll just make sure of it
# here's the order: 
# I want to first sort based on stim_id and then within stim_id 
# sort by rot_angle_deg
metadata.sort_values(by = ["stim_id", "rot_angle_deg"], ascending = True, inplace = True)
# metadata
png_paths = [f"{df_path}/{row['filename']}" for i, row in metadata.iterrows()]
# print(png_paths)
images = [Image.open(p).convert("RGB") for p in png_paths]

# here we need to make sure that all the views of an object are loaded first 
# before going to the next stimuli
print(png_paths)

# get how many unique stimuli we have?
n_stimuli = len(metadata["stim_id"].unique())
print(f"number of unique stimuli : {n_stimuli}")

# number of view points 
T = len(metadata["rot_angle_deg"].unique())
print(f"number fo viewpoints per object : {T}")

# now we preprocess these images before passing them to the encoder
# processor handles: resize (patch-aware), normalize, tensor conversion
inputs = processor(images=images, return_tensors="pt").to(device)

# make sure the model is frozen
encoder.eval()
for p in encoder.parameters():
    p.requires_grad = False

# now we pass the inputs to the encoder
with torch.no_grad():
    # pass the images to the encoder and get rid of cls tokens for now
    outputs = encoder(**inputs)
    embeddings = outputs.last_hidden_state[:, 1:, :]
    cls_embeddings = outputs.last_hidden_state[:, 0, :]
    # now reshape so that you have the different rotations of each stimulus as the second dim 

    out = cls_embeddings.view(n_stimuli, T, cls_embeddings.shape[-1])



out.shape


['/Users/lshahsha/Desktop/SM_new/png/x/polycubes_stim1_2_2_2_3_nc-9_rx0_var-base.png', '/Users/lshahsha/Desktop/SM_new/png/x/polycubes_stim1_2_2_2_3_nc-9_rx18_var-base.png', '/Users/lshahsha/Desktop/SM_new/png/x/polycubes_stim1_2_2_2_3_nc-9_rx37_var-base.png', '/Users/lshahsha/Desktop/SM_new/png/x/polycubes_stim1_2_2_2_3_nc-9_rx56_var-base.png', '/Users/lshahsha/Desktop/SM_new/png/x/polycubes_stim1_2_2_2_3_nc-9_rx75_var-base.png', '/Users/lshahsha/Desktop/SM_new/png/x/polycubes_stim1_2_2_2_3_nc-9_rx94_var-base.png', '/Users/lshahsha/Desktop/SM_new/png/x/polycubes_stim1_2_2_2_3_nc-9_rx113_var-base.png', '/Users/lshahsha/Desktop/SM_new/png/x/polycubes_stim1_2_2_2_3_nc-9_rx132_var-base.png', '/Users/lshahsha/Desktop/SM_new/png/x/polycubes_stim1_2_2_2_3_nc-9_rx151_var-base.png', '/Users/lshahsha/Desktop/SM_new/png/x/polycubes_stim1_2_2_2_3_nc-9_rx170_var-base.png', '/Users/lshahsha/Desktop/SM_new/png/x/polycubes_stim1_2_2_2_3_nc-9_rx189_var-base.png', '/Users/lshahsha/Desktop/SM_new/png/x/

torch.Size([5, 19, 384])

Now we need to make sense of the output of the lstm module:

For the doc ([Pytorch LSTM](https://docs.pytorch.org/docs/stable/generated/torch.nn.LSTM.html)):

The input is of dimension $(B, T, H_{in})$
- $B$: batch size
- $T$: sequence length
- $H_{in}$: input size/dimension

The output $(B, T, D*H_{out})$:
- $B$: batch size
- $T$: sequence length
- $H_{out}$: the output features. If proj_size > 0 then it will be proj_size, otherwise it will be hidden_size. It contains the output features ($h_t$) from the last layer of the LSTM, for each t. In other words, The output at every timestep. Think of it as the LSTM's "running commentary" — after seeing view 1, after seeing view 2, etc. Each of those T vectors is the **hidden state** at that moment in the sequence.

In [31]:
# now we pass these to an lstm
B, T, D = out.shape
hidden_size = 768
num_layers = 1
model = nn.LSTM(input_size = D, 
                hidden_size = hidden_size, 
                num_layers = num_layers, 
                bias = True, 
                batch_first = True, 
                dropout = 0.1, 
                bidirectional = False, 
                device = device)

out2, (hn, cn) = model(out)
print(out2.shape) # last hidden state? or all the hidden states
print(hn.shape) # hidden state 
print(cn.shape) # the conveyor belt 
print(out.shape)

torch.Size([5, 19, 768])
torch.Size([1, 5, 768])
torch.Size([1, 5, 768])
torch.Size([5, 19, 384])


/Users/lshahsha/Documents/GitHub/model_tutorial/.venv/lib/python3.11/site-packages/torch/nn/modules/rnn.py:990: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.1 and num_layers=1
  super().__init__("LSTM", *args, **kwargs)


In [32]:
# now we pass these to an lstm
# here we only pass the output of the Encoder
# but we can also pass the initial h (hidden state) and c (cell state)
# For nn.LSTM, h_0 and c_0 are always shaped as [num_layers * num_directions, batch_size, hidden_size]
# regardless of batch_first (that only affects the input/output, not the states).
# initializing them as all zeros:
B, T, D = out.shape
hidden_size = 768
num_layers = 1
# h0 = torch.zeros(1*num_layers, B, hidden_size)
# c0 = torch.zeros(1*num_layers, B, hidden_size)
# they can also be initialized to have random numbers
h0 = torch.randn(1*num_layers, B, hidden_size)
c0 = torch.randn(1*num_layers, B, hidden_size)
model = nn.LSTM(input_size = D, 
                hidden_size = hidden_size, 
                num_layers = num_layers, 
                bias = True, 
                batch_first = True, 
                dropout = 0.1, 
                bidirectional = False, 
                device = device)

out2, (hn, cn) = model(out)
print(out2.shape) # last hidden state? or all the hidden states
print(hn.shape) # hidden state 
print(cn.shape) # the conveyor belt 

torch.Size([5, 19, 768])
torch.Size([1, 5, 768])
torch.Size([1, 5, 768])


Now the question is: what do I want to train the model to do? that would define the loss function. The simplest case is: 

**Next-view prediction**: given views 1 to T-1, predict the DINO embedding of view T. So we can take the embeddings from 1 to T-1, and try to predict the embedding at time T (the last one). This could be a simple linear head that can project the embeddings to the embeddings of Dino. We can then compare this with the observed embedding at the final view. Concretely: 

- Take lstm_out[:, :-1, :] — the LSTM output at every timestep except the last, shape (N, T-1, hidden_size)
- Pass it through a small linear head to project back to DINO's embedding dimension: (N, T-1, 768) - this is supposed to predict all the timesteps embeddings from the output from the previous timestep
- Compare those predictions against the actual DINO embeddings at the next timestep embeddings[:, 1:, :] — shape (N, T-1, 768)


Here's a simple example to explain it:

Input to LSTM:

emb1 → emb2 → emb3 → emb4      # these are fed in sequentially

LSTM produces hidden states:

out1,  out2,  out3,  out4        # one per timestep, summarises everything seen so far


Then after the LSTM, you take those hidden states and use a linear head to make predictions:

- linear(out1) → predicted emb2
- linear(out2) → predicted emb3
- linear(out3) → predicted emb4

So the LSTM's job is just to accumulate information as it sees each view. The linear head's job is to use that accumulated information to guess what the next view's embedding will look like.

another way to look at this is this:

the output of lstm, the hidden states at each time point, is the  accumulated information up to and including that timepoint:

out1 = "I have seen emb1"
out2 = "I have seen emb1, emb2"
out3 = "I have seen emb1, emb2, emb3"
out4 = "I have seen emb1, emb2, emb3, emb4"

So lstm_out is not a sequence of predictions — it's a sequence of summaries, each one richer than the last as more views are seen. And with the linear predictor head we are trying to use that accumulated information up to time point X to predict the embeddings at time point X. 

In [10]:
# # now let's build an LSTM 

# class myLSTM(nn.Module):
#     def __init__(self, input_dim, hidden_dim, output_dim):
#         super().__init__():

#         self.input_dim = input_dim
#         self.hidden_dim = hidden_dim
#         self.output_dim = output_dim

#         self.lstm = nn.LSTM(input_size = self.input_dim, 
#                             hidden_size = self.hidden_dim, 
#                             num_layers = 1, 
#                             bias = True, 
#                             batch_first = True, 
#                             dropout = 0.1, 
#                             bidirectional = False, 
#                             device = device)

#         # need to think what the readout should be
#         # self.readout = 

#     def forward(self, x):
#         output, (hn, cn) = self.lstm(x)
#         return output, (hn, cn)

In [34]:
# get the accumulated information from 0 to T-1
tmp_out = out2[:, :-1, :]
print(tmp_out.shape)
# get the embeddings 
tmp_embeddings = out[:, 1:, :]
print(tmp_embeddings.shape)

torch.Size([5, 18, 768])
torch.Size([5, 18, 384])


In [35]:
class LinearHead(nn.Module):
    def __init__(self, hidden_dim, out_dim):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.out_dim = out_dim

        self.fc = nn.Linear(hidden_dim, out_dim, bias = False)

    def forward(self, x):
        y = self.fc(x)
        return y 

In [36]:
linhead = LinearHead(hidden_dim=tmp_out.shape[-1], out_dim=tmp_embeddings.shape[-1])

y = linhead(tmp_out)

In [37]:
y.shape

torch.Size([5, 18, 384])

In [48]:
opt     = optim.Adam(model.parameters(), lr=1e-3)
# now we need to compute the similarity between pairs along the embedding dimension
# so we set the dim = -1
loss_fn = 1 - F.cosine_similarity(y, tmp_embeddings, dim = -1)
loss_fn

tensor([[0.9332, 0.9363, 0.9074, 0.9024, 0.8862, 0.9081, 0.9292, 0.9144, 0.9166,
         0.9260, 0.9564, 0.9458, 0.9853, 0.9334, 0.9425, 0.9566, 0.9266, 0.9486],
        [0.9338, 0.9540, 0.9360, 0.9153, 0.8764, 0.9274, 0.9611, 0.9382, 0.9261,
         0.9042, 0.9357, 0.9178, 0.9405, 0.9361, 0.9347, 0.9424, 0.9265, 0.9385],
        [0.9301, 0.9567, 0.9319, 0.9293, 0.9025, 0.9402, 0.9582, 0.9390, 0.9056,
         0.9118, 0.9359, 0.8922, 0.9205, 0.8703, 0.9115, 0.9446, 0.9140, 0.9313],
        [0.8947, 0.9143, 0.9019, 0.9208, 0.9467, 0.9490, 0.9457, 0.9188, 0.9050,
         0.9015, 0.9338, 0.9292, 0.9701, 0.9297, 0.9298, 0.9349, 0.9260, 0.9198],
        [0.9378, 0.9678, 0.9316, 0.9378, 0.9316, 0.9516, 0.9504, 0.9169, 0.9121,
         0.9131, 0.9061, 0.9109, 0.9218, 0.8811, 0.8803, 0.9326, 0.9021, 0.9343]],
       grad_fn=<RsubBackward1>)

Now putting it all together: 

- 

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image


import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, random_split
from torch.utils.data import DataLoader 

from transformers import AutoImageProcessor, AutoModel




# first define the dataset class
# Data needs to be stored in a specific directory structure 
# parent_dir/objects/object_instances/instance_rotations

# the dictionary containing model info
config = {
    "facebook/dinov2-small": {"dim": 384},
    "facebook/dinov2-base":  {"dim": 768},
}

class DIEBenchDataset(Dataset):
    def __init__(self, root, model_name="facebook/dinov2-small"):
        self.processor = AutoImageProcessor.from_pretrained(model_name)
        self.object_dirs = sorted([
            obj_dir
            for synset_dir in sorted(Path(root).iterdir()) if synset_dir.is_dir()
            for obj_dir in sorted(synset_dir.iterdir()) if obj_dir.is_dir()
        ])

    def __len__(self):
        return len(self.object_dirs)

    def __getitem__(self, idx):
        obj_dir = self.object_dirs[idx]
        image_paths = sorted(obj_dir.glob("image_*.jpg"))
        images = [Image.open(p).convert("RGB") for p in image_paths]
        pixel_values = self.processor(images=images, return_tensors="pt")["pixel_values"]
        return pixel_values  # (T, C, H, W)


class MRSimple(nn.Module):
    def __init__(self,
                 hidden_dim,
                 encoder_variant="facebook/dinov2-small",
                 dropout=0.1,
                 lstm_num_layers=1):
        super().__init__()

        self.hidden_dim    = hidden_dim
        self.embedding_dim = config[encoder_variant]["dim"]

        self.encoder = AutoModel.from_pretrained(encoder_variant)
        self._freeze_encoder()

        lstm_dropout = dropout if lstm_num_layers > 1 else 0.0
        self.lstm = nn.LSTM(
            input_size    = self.embedding_dim,
            hidden_size   = self.hidden_dim,
            num_layers    = lstm_num_layers,
            bias          = True,
            batch_first   = True,
            dropout       = lstm_dropout,
            bidirectional = False,
        )
        self.fc = nn.Linear(self.hidden_dim, self.embedding_dim, bias=False)

    def _freeze_encoder(self):
        self.encoder.eval()
        for p in self.encoder.parameters():
            p.requires_grad = False

    def forward(self, images):
        # images: (B, T, C, H, W) — already preprocessed tensors from the dataset
        B, T, C, H, W = images.shape
        print(f"images shape is {images.shape}")

        # flatten batch and time so encoder sees (B*T, C, H, W)
        # to understand what happens here, take a look at this example:
        # images[0, 0]  → object 1, timestep 1
        # images[0, 1]  → object 1, timestep 2
        # images[0, 2]  → object 1, timestep 3
        # images[1, 0]  → object 2, timestep 1
        # images[1, 1]  → object 2, timestep 2
        # images[1, 2]  → object 2, timestep 3
        # after flattening:
        # flat[0]  → object 1, timestep 1
        # flat[1]  → object 1, timestep 2
        # flat[2]  → object 1, timestep 3
        # flat[3]  → object 2, timestep 1
        # flat[4]  → object 2, timestep 2
        # flat[5]  → object 2, timestep 3
        # this is essentally the same as running a loop over all the time steps and then concatenating
        flat = images.view(B * T, C, H, W)

        with torch.no_grad():
            outputs        = self.encoder(pixel_values=flat)
            # get the cls tokens
            # in case you also want the embeddings
            # embeddings = outputs.last_hidden_state[:, 1:, :]
            cls_embeddings = outputs.last_hidden_state[:, 0, :]  # (B*T, embedding_dim)

        # reshape back to (B, T, embedding_dim)
        # this needs to be done before passing on to the lstm
        out = cls_embeddings.view(B, T, self.embedding_dim)

        # lstm
        hidden, (hn, cn) = self.lstm(out)              # (B, T, hidden_dim)


        # why we do this?
        # Input to LSTM:
        # emb1 → emb2 → emb3 → emb4      # these are fed in sequentially
        # LSTM produces hidden states:
        # out1,  out2,  out3,  out4        # one per timestep, summarises everything seen so far
        # Then after the LSTM, you take those hidden states and use a linear head to make predictions:
        # - linear(out1) → predicted emb2
        # - linear(out2) → predicted emb3
        # - linear(out3) → predicted emb4
        # So the LSTM's job is just to accumulate information as it sees each view. 
        # The linear head's job is to use that accumulated information to guess 
        # what the next view's embedding will look like.
        # another way to look at this is this:
        # the output of lstm, the hidden states at each time point, 
        # is the  accumulated information up to and including that timepoint:
        # out1 = "I have seen emb1"
        # out2 = "I have seen emb1, emb2"
        # out3 = "I have seen emb1, emb2, emb3"
        # out4 = "I have seen emb1, emb2, emb3, emb4"

        # So lstm_out is not a sequence of predictions — 
        # it's a sequence of summaries, each one richer than 
        # the last as more views are seen. And with the linear 
        # predictor head we are trying to use that accumulated i
        # nformation up to time point X to predict the embeddings at time point X. 

        # predictions are all but last step
        predictions = self.fc(hidden[:, :-1, :])       # (B, T-1, embedding_dim)
        # shift the targets by one
        targets     = out[:, 1:, :].detach()           # (B, T-1, embedding_dim)

        return predictions, targets
    



def train(root):
    HIDDEN_DIM = 512
    ENCODER_VARIANT = "facebook/dinov2-small"
    LSTM_NUM_LAYERS = 1
    DROPOUT = 0.1
    BATCH_SIZE = 128
    MAX_EPOCH = 1000
    # get the device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # create an isntance of the model
    model = MRSimple(
        hidden_dim      = HIDDEN_DIM, 
        encoder_variant = ENCODER_VARIANT, 
        dropout         = DROPOUT, 
        lstm_num_layers = LSTM_NUM_LAYERS,
    ).to(device)

    # we will do a cross validation style
    # for now, just using one train set and one test set
    # create an instance of the 

    # define the optimizer
    optimizer = optim.Adam(model.parameters(), lr=1e-3)

    # record loss
    loss_all = []

    # now create the dataset class
    dataset = DIEBenchDataset(root, model_name=ENCODER_VARIANT)

    # now let's divide to train and test sets
    train_size = int(0.8 * len(dataset))
    test_size = len(dataset) - train_size

    # divide to train and test using random split
    train_set, test_set = random_split(dataset, [train_size, test_size])

    # now make loader objects
    train_loader = DataLoader(
                            train_set,
                            batch_size=BATCH_SIZE,
                            shuffle=False,
                            sampler=None,
                            batch_sampler=None,
                            num_workers=0, # for parallel loading?
                            # collate_fn=stack_fn, # THIS IS IMPORTANT
                            drop_last=False,
                            pin_memory=False,
                            persistent_workers=False,
                        )
    test_loader = DataLoader(
                            test_set,
                            batch_size=BATCH_SIZE,
                            shuffle=False,
                            sampler=None,
                            batch_sampler=None,
                            num_workers=0, # for parallel loading?
                            # collate_fn=stack_fn, # THIS IS IMPORTANT
                            drop_last=False,
                            pin_memory=False,
                            persistent_workers=False,
                        )
    # define an optimizer
    optimizer = optim.Adam(model.parameters(), lr=1e-3)

    # now training
    loss_all = [] # to store all the loss values
    # put the model in training mode
    model.train()
    for epoch, images in enumerate(train_loader):
        # first make sure images and labels are on the correct device
        images = images.to(device) # Move images to device

        # firsts pass the images to the model we instantiated earlier
        emb_pred, targets = model(images)# Forward pass: compute predicted embeddings

        # loss is defined as the 1 - cosine similarity across the embedding
        current_loss = 1 - F.cosine_similarity(emb_pred, targets, dim = -1).mean()
        # store the current value of loss
        loss_all.append(current_loss.item())


        # clear out the optimizer gradients
        optimizer.zero_grad()

        # backprop
        current_loss.backward()

        # update the parameters
        optimizer.step()

        if epoch % 10 == 0:
            print(f"epoch {epoch} | loss: {current_loss.item():.4f}")

    # once training is done, test:
    # first put the model in eval mode
    model.eval()

    dis_test = []
    for epoch, images in enumerate(test_loader):
        # first make sure images and labels are on the correct device
        images = images.to(device) # Move images to device
        # firsts pass the images to the model we instantiated earlier
        emb_pred_test, targets_test = model(images)# Forward pass: compute predicted embeddings
        # calculate distance 1 - cosine similarity

        # loss is defined as the 1 - cosine similarity across the embedding
        current_dist = 1 - F.cosine_similarity(emb_pred_test, targets_test, dim = -1).mean()

        # record it
        dis_test.append(current_dist.item())

    # printout the final test score?
    print(np.mean(dis_test))
    return model, loss_all




train(root_dir)